In [30]:
!pip install wikipedia requests

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=8192fa6da0a18c3cb7b553b56db6a57efc00837607c76f995f6d587e621093a5
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [31]:
import requests
import wikipedia

# Placeholder for your OpenWeatherMap API key
OPENWEATHER_API_KEY = "37a71b2d436b0d6a00c225a3534870ce"  # Replace with your key later

In [32]:
def weather_agent(city):
    """
    Takes a city name and returns current weather info.
    """
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"

    try:
        response = requests.get(url)
        data = response.json()

        if data.get("cod") != 200:
            return f"Sorry, could not find weather for {city}."

        weather_desc = data["weather"][0]["description"].title()
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]

        return f"Weather in {city}:\n- Temperature: {temp}°C\n- Condition: {weather_desc}\n- Humidity: {humidity}%"

    except Exception as e:
        return f"Error: {e}"

In [33]:
print(weather_agent ("Coimbatore"))

Weather in Coimbatore:
- Temperature: 23.88°C
- Condition: Light Intensity Drizzle
- Humidity: 88%


In [34]:
import requests

city = "Chennai"
url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
response = requests.get(url).json()
print(response)

{'coord': {'lon': 80.2785, 'lat': 13.0878}, 'weather': [{'id': 701, 'main': 'Mist', 'description': 'mist', 'icon': '50n'}], 'base': 'stations', 'main': {'temp': 23.18, 'feels_like': 24.03, 'temp_min': 22.78, 'temp_max': 23.99, 'pressure': 1013, 'humidity': 95, 'sea_level': 1013, 'grnd_level': 1012}, 'visibility': 2500, 'wind': {'speed': 2.06, 'deg': 340}, 'clouds': {'all': 75}, 'dt': 1763489248, 'sys': {'type': 2, 'id': 2104103, 'country': 'IN', 'sunrise': 1763426348, 'sunset': 1763467752}, 'timezone': 19800, 'id': 1264527, 'name': 'Chennai', 'cod': 200}


In [35]:
def wiki_agent(query):
    """
    Takes a query string and returns a short Wikipedia summary.
    """
    try:
        summary = wikipedia.summary(query, sentences=2)  # 2 sentences summary
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Your query is ambiguous. Did you mean: {e.options[:5]}?"
    except wikipedia.exceptions.PageError:
        return "Sorry, no Wikipedia page found for your query."
    except Exception as e:
        return f"Error: {e}"

In [36]:
print(wiki_agent("Thalapathy Vijay"))

Joseph Vijay Chandrasekhar (born 22 June 1974), known professionally as Vijay, is an Indian politician and actor. In a career spanning over three decades, Vijay acted in 68 films and is one of the most commercially successful actors in Tamil cinema with multiple films amongst the highest-grossing Tamil films of all time and is amongst the highest paid actors in India.


In [37]:
def controller(query):
    query_lower = query.lower()

    if "weather in" in query_lower:
        # Extract city name
        city = query_lower.replace("weather in", "").strip()
        return weather_agent(city)
    else:
        return wiki_agent(query)

In [47]:
print(controller("weather in Coimbatore"))
print(controller("weather in chennai"))
print(controller("who is Thalapathy Vijay"))
print(controller("who is Elon Musk"))

Weather in coimbatore:
- Temperature: 23.88°C
- Condition: Light Intensity Drizzle
- Humidity: 88%
Weather in chennai:
- Temperature: 23.18°C
- Condition: Mist
- Humidity: 95%
Joseph Vijay Chandrasekhar (born 22 June 1974), known professionally as Vijay, is an Indian politician and actor. In a career spanning over three decades, Vijay acted in 68 films and is one of the most commercially successful actors in Tamil cinema with multiple films amongst the highest-grossing Tamil films of all time and is amongst the highest paid actors in India.
Elon Reeve Musk (born June 28, 1971) is a businessman and entrepreneur known for his leadership of Tesla, SpaceX, Twitter, and xAI. Musk has been the wealthiest person in the world since 2021; as of October 2025, Forbes estimates his net worth to be around $500 billion.
Born into a wealthy family in Pretoria, South Africa, Musk emigrated in 1989 to Canada; his Canadian citizenship is congenital, his mother having been born there.


In [39]:
!pip install streamlit

In [40]:
api_key = "37a71b2d436b0d6a00c225a3534870ce"

In [41]:
%%writefile app.py
import streamlit as st
import requests
from bs4 import BeautifulSoup

# ---------------------- WEATHER AGENT ----------------------
def get_weather(city):
    api_key = "YOUR_API_KEY"   # Replace with your real API key
    base_url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&units=metric&appid={api_key}"

    try:
        response = requests.get(base_url)
        data = response.json()

        if response.status_code == 200:
            description = data['weather'][0]['description'].title()
            temperature = data['main']['temp']
            humidity = data['main']['humidity']

            return f"🌦 *Weather in {city.title()}*\n\n" \
                   f"- Condition: {description}\n" \
                   f"- Temperature: {temperature}°C\n" \
                   f"- Humidity: {humidity}%"

        else:
            return "❌ Sorry, could not find weather details."

    except Exception:
        return "⚠ Error fetching weather data."


# ---------------------- WIKI AGENT ----------------------
def get_wiki_answer(query):
    try:
        url = f"https://en.wikipedia.org/wiki/{query.replace(' ', '_')}"
        page = requests.get(url)
        soup = BeautifulSoup(page.content, "html.parser")
        paragraphs = soup.find_all("p")

        if paragraphs:
            summary = paragraphs[1].text.strip()
            return f"📘 *Summary for {query.title()}*\n\n{summary}"

        return "❌ No Wikipedia info found."

    except Exception:
        return "⚠ Error retrieving Wikipedia details."


# ---------------------- CONTROLLER ----------------------
def controller(user_query):
    user_query_lower = user_query.lower()

    if "weather" in user_query_lower:
        city = user_query_lower.replace("weather in", "").strip()
        return get_weather(city)

    elif user_query:
        return get_wiki_answer(user_query)

    return "❌ Invalid query."


# ---------------------- STREAMLIT UI ----------------------
st.set_page_config(
    page_title="AI Dual Agent",
    page_icon="🤖",
    layout="centered",
    initial_sidebar_state="collapsed"
)

st.markdown(
    """
    <h1 style='text-align: center; color: white;'>🤖 AI Dual Agent</h1>
    <p style='text-align: center; color: #bbbbbb;'>Your Smart Assistant for Weather & Wikipedia</p>
    """,
    unsafe_allow_html=True
)

st.markdown("---")

st.markdown("### 🔍 Ask me anything!")

user_input = st.text_input("Enter your question (e.g., 'Weather in Chennai', 'Albert Einstein')")

if st.button("Get Answer"):
    if user_input.strip():
        result = controller(user_input)
        st.markdown("### 📌 Result:")
        st.write(result)
    else:
        st.warning("Please enter a question.")

# Dark mode background
st.markdown(
    """
    <style>
    body {
        background-color: #111 !important;
        color: white;
    }
    </style>
    """,
    unsafe_allow_html=True
)

Overwriting app.py


In [42]:
!pip install pyngrok

In [43]:
from pyngrok import ngrok

# Paste your new key here
ngrok.set_auth_token("35f39ljoIHvsTfEiLv5P9r3jUEE_7Psdc1WTRr7ukUt651Ni7")

In [44]:
ngrok.kill()

In [45]:
public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")

Your app is live at: NgrokTunnel: "https://lifeful-adelaide-vermicularly.ngrok-free.dev" -> "http://localhost:8501"
